In [ ]:

"""Recompute monthly BMKG vs. Open-Meteo validation metrics."""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, norm
from IPython.display import display

BASE_DIR = Path.cwd()
if BASE_DIR.name == "validation_monthly":
    pass
else:
    BASE_DIR = BASE_DIR / "validation_monthly"

INPUT_FILE = BASE_DIR / "bmkg_openmeteo_monthly_2025.csv"

VARIABLES = {
    "rainfall": ("rainfall", "om_rainfall"),
    "hum_avg": ("hum_avg", "om_hum_avg"),
    "temp_max": ("temp_max", "om_temp_max"),
    "temp_min": ("temp_min", "om_temp_min"),
    "temp_avg": ("temp_avg", "om_temp_avg"),
}

CONFIDENCE_LEVEL = 0.95
METRICS = [
    "bias",
    "MAE",
    "RMSE",
    "Pearson_r",
    "Pearson_CI_lower",
    "Pearson_CI_upper",
    "p_value",
]


def calculate_metrics(data):
    """Calculate validation metrics for each station and variable."""

    results = []

    alpha = 1 - CONFIDENCE_LEVEL
    z_critical = norm.ppf(1 - alpha / 2)

    for (station_id, station_name), group in data.groupby(
        ["station_id", "station_name"], sort=True
    ):
        group = group.sort_values("month")

        for variable, (bmkg_col, om_col) in VARIABLES.items():
            pair = group[[bmkg_col, om_col]].dropna()

            bmkg = pair[bmkg_col].to_numpy(dtype=float)
            om = pair[om_col].to_numpy(dtype=float)
            n = len(pair)

            if n < 4:
                raise ValueError(
                    f"At least four paired observations are required "
                    f"for Pearson CI: {station_name}, {variable}."
                )

            if np.std(bmkg) == 0 or np.std(om) == 0:
                raise ValueError(
                    f"Correlation cannot be calculated because one "
                    f"variable is constant: {station_name}, {variable}."
                )

            correlation = pearsonr(bmkg, om)
            r = correlation.statistic

            # Fisher z-transformation for the Pearson correlation CI.
            r_clipped = np.clip(r, -0.999999, 0.999999)
            fisher_z = np.arctanh(r_clipped)
            standard_error = 1 / np.sqrt(n - 3)

            ci_lower = np.tanh(
                fisher_z - z_critical * standard_error
            )
            ci_upper = np.tanh(
                fisher_z + z_critical * standard_error
            )

            results.append({
                "station_id": station_id,
                "station_name": station_name,
                "variable": variable,
                "n_months": n,
                "bias": np.mean(om - bmkg),
                "MAE": np.mean(np.abs(om - bmkg)),
                "RMSE": np.sqrt(np.mean((om - bmkg) ** 2)),
                "Pearson_r": r,
                "Pearson_CI_lower": ci_lower,
                "Pearson_CI_upper": ci_upper,
                "p_value": correlation.pvalue,
            })

    return pd.DataFrame(results)


# Load monthly BMKG and Open-Meteo observations.
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

data = pd.read_csv(INPUT_FILE)

required_columns = {
    "month",
    "station_id",
    "station_name",
    *[column for pair in VARIABLES.values() for column in pair],
}

missing = required_columns - set(data.columns)

if missing:
    raise ValueError(
        f"Required input columns are missing: {sorted(missing)}"
    )

# Calculate and display validation metrics.
results = calculate_metrics(data)

print(f"Number of monthly records: {len(data)}")
print(f"Number of validation results: {len(results)}")
print(f"Pearson correlation confidence level: {CONFIDENCE_LEVEL:.0%}")

display(
    results.style.format({
        "bias": "{:.4f}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "Pearson_r": "{:.4f}",
        "Pearson_CI_lower": "{:.4f}",
        "Pearson_CI_upper": "{:.4f}",
        "p_value": "{:.6g}",
    })
)
